### 발표 평가 요소 구분 모듈 -- 자연어

#### 환경 설정

In [1]:
# library import
import os
from dotenv import load_dotenv
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 환경 변수
load_dotenv()
BASE_URL = os.getenv("OPENAI_BASE_URL")
API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("OPENAI_MODEL")

#### Structured Output 정의

In [2]:
# 발표 평가 요소 모델 정의
# 주의: json_schema strict 모드는 dict[str, ...]처럼 키가 고정되지 않은 필드를 지원하지 않으므로
# key/value를 명시적 필드로 갖는 항목 리스트로 표현
class EvaluableItem(BaseModel):
	key: str = Field(description="평가 요소 이름")
	value: str | bool | int | float | None = Field(description="평가 요소에 대한 값")

class NeedMoreInfoItem(BaseModel):
	key: str = Field(description="평가 요소 이름")
	value: str = Field(description="현재까지 확인된 값 (구체적인 기준 미포함)")

class EvaluationCriteriaAnalysis(BaseModel):
	evaluable: list[EvaluableItem] = Field(description="발표 평가에 적용할 요소와 그 값")
	need_more_info: list[NeedMoreInfoItem] = Field(description="발표 평가에 적용하기 위해 정보가 더 필요한 값")
	not_evaluable: list[str] = Field(description="발표 평가에 적용하지 않을 요소")

#### System Prompt 작성

In [3]:
SYSTEM_PROMPT = """
너는 발표 평가 기준을 분석하는 전문가야.

입력은 사용자가 자유로운 문장으로 작성한 발표 평가 기준이야.
입력을 분석하여 각 평가 기준을 다음 세 가지로 구분해야 해.

## 1. 판단할 값 (evaluable)

현재 발표 분석 시스템에서 실제로 측정하거나 판단할 수 있고,
평가에 필요한 정보가 충분히 주어진 평가 기준.

현재 시스템에서 판단할 수 있는 요소는 다음과 같아.

* 발표 시간
* 말하기 속도
* 발음
* 억양
* 음량
* 멈춤/침묵
* 필러
* 시선 처리
* 대본 사용 여부
* 대본 의존도
* 실제 발화와 대본의 의미적 유사도

판단할 값은 `key: value` 형태로 추출해.

예시:

* "대본은 보면 안 돼."
  → `"대본 사용": false`
* "발표 시간은 10분 이내여야 한다."
  → `"발표 시간": "10분 이내"`

## 2. 정보가 더 필요한 값 (need_more_info)

현재 발표 분석 시스템에서 판단할 수 있는 요소이지만,
입력에 평가에 필요한 구체적인 정보가 충분히 주어지지 않은 경우.

입력에서 확인할 수 있는 평가 요소와 현재 주어진 값은 그대로 추출하되,
구체적인 기준이 부족한 경우 `need_more_info`에 포함해.

예시:

* "발표 시간을 엄수해야 한다."
  → `"발표 시간": "엄수"`
* "말하는 속도를 적절하게 유지해야 한다."
  → `"말하기 속도": "적절하게"`

입력에 없는 구체적인 수치나 기준을 임의로 생성하지 마.

## 3. 판단하지 않을 값 (not_evaluable)

현재 발표 분석 시스템에서 측정하거나 판단하지 않는 평가 요소.

`not_evaluable`에는 평가 요소의 이름만 추출하고,
해당 요소의 구체적인 값은 추출하지 마.

예시:

* "발표 주제에는 Agent성이 잘 드러나야 한다."
  → `"발표 주제"`
* "다른 팀의 주제와 유사하면 안 된다."
  → `"주제 유사도"`
* "발표자의 용모가 단정해야 한다."
  → `"용모"`
* "질의응답을 잘해야 한다."
  → `"질의응답"`

현재 판단하지 않는 주요 요소는 다음과 같아.

* 발표 주제 선정 및 적절성
* 발표 내용의 신뢰도
* 시각 자료를 활용한 내용의 신뢰도
* 사용자의 용모 등 외적 요소 (시선 처리는 제외)
* 질의응답

## 분류 규칙

* 먼저 해당 평가 기준이 현재 발표 분석 시스템에서 판단 가능한 요소인지 판단해.
* 판단 가능한 요소이며 평가에 필요한 정보가 충분하면 `evaluable`에 포함해.
* 판단 가능한 요소이지만 구체적인 정보가 부족하면 `need_more_info`에 포함해.
* 현재 시스템에서 판단할 수 없는 요소는 `not_evaluable`에 포함해.
* `need_more_info`는 평가 자체가 불가능하다는 의미가 아니라, 추가 정보가 있으면 `evaluable`로 처리할 수 있는 요소를 의미해.
* `not_evaluable`은 추가 정보를 받아도 현재 시스템에서 평가하지 않는 요소를 의미해.
* 입력에 명시되지 않은 값이나 구체적인 수치, 기준을 임의로 생성하지 마.
* 동일한 평가 요소가 여러 번 등장하는 경우 하나로 통합해.
* 입력의 의미를 유지하면서 평가 요소를 적절한 key 이름으로 정리해.
* `not_evaluable`에는 요소의 이름만 포함하고 세부 값은 포함하지 마.


항상 evaluable, need_more_info, not_evaluable 세 필드를 모두 반환해야 해.
해당하는 평가 요소가 없는 경우 빈 객체({}) 또는 빈 리스트([])를 사용해.
필드 자체를 생략하면 안 돼.
"""

#### LLM 모델 불러오기

In [4]:
os.getenv('OPENAI_MODEL'), os.getenv('OPENAI_BASE_URL')

('openai/gpt-5.6-luna',
 'https://mlapi.run/286e9158-d32e-436d-a23d-36b43fc8e68a/v1')

In [5]:
# LLM 객체 생성
model = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
)

# Structured Output 연결
# evaluable/need_more_info를 key/value 리스트로 바꿔 자유 키 dict를 제거했으므로
# json_schema strict 모드(기본값)를 그대로 사용 가능 (function tools가 아니라 response_format 기반이라
# reasoning_effort와 충돌하지 않음)
structured_output = model.with_structured_output(EvaluationCriteriaAnalysis)

# 실제 모델 호출이 되는지 간단한 테스트
test_input = "발표는 10분 정도 진행하며 시간을 지켜야 합니다. 발표 중 불필요한 추임새를 줄이고 적절한 음량과 억양을 사용해야 합니다. 발표 자료의 내용은 신뢰할 수 있어야 하고 질의응답에도 적극적으로 참여해야 합니다."
response = structured_output.invoke(test_input)
response

EvaluationCriteriaAnalysis(evaluable=[], need_more_info=[NeedMoreInfoItem(key='발표 시간 준수', value='발표가 실제로 약 10분 동안 진행되었는지 확인할 정보가 없습니다.'), NeedMoreInfoItem(key='불필요한 추임새', value='발표 중 불필요한 추임새의 빈도와 정도를 확인할 정보가 없습니다.'), NeedMoreInfoItem(key='음량과 억양', value='발표자의 음량이 적절했는지, 억양을 효과적으로 사용했는지 확인할 정보가 없습니다.'), NeedMoreInfoItem(key='발표 자료의 신뢰성', value='발표 자료의 출처와 내용의 정확성을 검토할 정보가 없습니다.'), NeedMoreInfoItem(key='질의응답 참여', value='질의응답에서 질문에 적극적으로 참여하고 적절히 답변했는지 확인할 정보가 없습니다.')], not_evaluable=[])

#### 단일 평가 기준 test

In [6]:
criteria_input = "발표 시간은 10분이며 시간을 지켜야 합니다. 발표 중 불필요한 추임새를 줄이고 적절한 음량과 억양을 사용해야 합니다. 발표 자료의 내용은 신뢰할 수 있어야 하고 질의응답에도 적극적으로 참여해야 합니다."

result = structured_output.invoke(
    [
        ("system", SYSTEM_PROMPT),
        ("user", criteria_input)
    ]
)
data = result.model_dump()
data

{'evaluable': [{'key': '발표 시간', 'value': '10분'}],
 'need_more_info': [{'key': '필러', 'value': '불필요한 추임새를 줄여야 함'},
  {'key': '음량', 'value': '적절한 음량'},
  {'key': '억양', 'value': '적절한 억양'}],
 'not_evaluable': ['발표 자료 내용의 신뢰도', '질의응답']}

In [7]:
criteria_inputs = ["시간은 5분이며 ±30초를 허용합니다. 대본은 사용하지 않아야 하고, 발표자의 시선은 청중을 향해야 합니다.",
"발표는 시간을 엄수해야 하며, 말하는 속도도 너무 빠르지 않도록 주의해야 합니다. 발표 내용은 이해하기 쉽고 논리적으로 구성되어야 합니다.",
"발표 시간은 10분 이내로 해야 합니다. 대본을 보면서 발표해도 되지만, 지나치게 의존해서는 안 됩니다. 발표자의 자세도 단정해야 합니다.",
"발표 주제는 창의적이어야 하며 우리 팀의 Agent 활용이 잘 드러나야 합니다. 다른 팀과 비슷한 주제는 피해야 합니다. 발표 시간은 7분입니다.",
"발표 시간은 10분이며 시간을 지켜야 합니다. 발표 중 불필요한 추임새를 줄이고 적절한 음량과 억양을 사용해야 합니다. 발표 자료의 내용은 신뢰할 수 있어야 하고 질의응답에도 적극적으로 참여해야 합니다."
]

for i in range(len(criteria_inputs)):
    result = structured_output.invoke(
        [
            ("system", SYSTEM_PROMPT),
            ("user", criteria_inputs[i])
        ]
    )
    data = result.model_dump()
    print(f"Input {i+1}: {criteria_inputs[i]}")
    print(f"Output {i+1}: {data}\n")

Input 1: 시간은 5분이며 ±30초를 허용합니다. 대본은 사용하지 않아야 하고, 발표자의 시선은 청중을 향해야 합니다.
Output 1: {'evaluable': [{'key': '발표 시간', 'value': '5분, ±30초 허용'}, {'key': '대본 사용', 'value': False}, {'key': '시선 처리', 'value': '청중을 향해야 함'}], 'need_more_info': [], 'not_evaluable': []}

Input 2: 발표는 시간을 엄수해야 하며, 말하는 속도도 너무 빠르지 않도록 주의해야 합니다. 발표 내용은 이해하기 쉽고 논리적으로 구성되어야 합니다.
Output 2: {'evaluable': [], 'need_more_info': [{'key': '발표 시간', 'value': '엄수해야 함'}, {'key': '말하기 속도', 'value': '너무 빠르지 않도록 해야 함'}], 'not_evaluable': ['발표 내용의 이해 용이성', '발표 내용의 논리적 구성']}

Input 3: 발표 시간은 10분 이내로 해야 합니다. 대본을 보면서 발표해도 되지만, 지나치게 의존해서는 안 됩니다. 발표자의 자세도 단정해야 합니다.
Output 3: {'evaluable': [{'key': '발표 시간', 'value': '10분 이내'}, {'key': '대본 사용', 'value': True}], 'need_more_info': [{'key': '대본 의존도', 'value': '지나치게 의존해서는 안 됨'}], 'not_evaluable': ['자세']}

Input 4: 발표 주제는 창의적이어야 하며 우리 팀의 Agent 활용이 잘 드러나야 합니다. 다른 팀과 비슷한 주제는 피해야 합니다. 발표 시간은 7분입니다.
Output 4: {'evaluable': [{'key': '발표 시간', 'value': '7분'}], 'need_more_info': [], 'not_evaluable': ['발표 주제